# LLM08: Mixture of Experts (MoE) & Numerical Precision

## Lab Overview

1. **Mixture of Experts (MoE)**: A sparsely activated architecture that increases model capacity by using multiple experts, while activating only a small subset of them for each token. Many modern LLMs use MoE-style routing to improve the capacity/compute trade-off.
2. **Numerical Precision & Quantization**: Understanding common numerical formats used in LLMs (FP32, BF16, FP16, and conceptually FP8), and how quantization (such as INT8/INT4) can reduce memory footprint and often improve inference efficiency.

> **Quick term:** **MoE (Mixture of Experts)** means multiple separate “expert” feed-forward networks plus a **router** that picks which expert(s) process each token. Only a few experts activate per token, so compute stays much closer to a small model while total parameter count can be much larger.

#### Recommended Hardware

AMD Ryzen™ AI Halo Processors (e.g., AI Max+ 395, AI Max 390)

#### Software Environment

OS: Ubuntu 24.04.3 LTS \
Install [AUP Learning Cloud](https://amdresearch.github.io/aup-learning-cloud/installation/quick-start.html?family=ryzen-ai&gpu=…). After installing AUP Learning Cloud, you will have a ROCm and PyTorch environment that is compatible with this notebook.

## Goals

By the end of this lab, you will be able to:

1. Understand the MoE architecture: expert networks, router/gating mechanism, and sparse activation.
2. Implement Top-K gating and expert routing from scratch.
3. Explore why load balancing matters in MoE training.
4. Compare floating-point formats (FP32, FP16, BF16, and conceptually FP8) and their trade-offs.
5. Implement basic symmetric quantization (INT8/INT4 simulation) and measure its impact on model weights and outputs.


## 1. Environment Setup


In [6]:
import math
import warnings

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

torch.manual_seed(42)
np.random.seed(42)

Using device: cuda
PyTorch version: 2.9.1+rocm7.2.0.git7e1940d4
GPU: AMD Radeon(TM) 8060S Graphics
GPU Memory: 102.7 GB


## 2. MoE Concept

The Mixture of Experts idea is simple: instead of one large FFN, use **multiple smaller FFNs** ("experts") and a **router** ("gating network") that decides which expert(s) to activate for each input token.

A dense formulation is:

$$
y = \sum_{i=1}^{M} G(x)_i \cdot E_i(x),
$$

where $E_i$ is expert $i$ and $G(x)_i$ is its routing weight.

In sparse MoE, we first compute router logits

$$
r = xW_g,
$$

then select the **top-$k$ experts** for each token, and apply softmax **only over those selected experts**. This produces a sparse gate: most experts get weight 0, and only $k$ experts have nonzero weights.

**Why this is useful:**

- **Model capacity** scales with the number of experts $M$
- **Per-token compute** stays much closer to using only $k$ experts rather than all experts

> In practice, the actual speedup depends not only on $k$, but also on routing, token dispatch, memory access, and distributed communication overhead.

**Notable MoE examples (illustrative):**

| Model           | #Experts     | Activated Experts           | Notes                         |
| --------------- | ------------ | --------------------------- | ----------------------------- |
| Mixtral 8×7B    | 8            | Top-2                       | Popular open-weight MoE model |
| Mixtral 8×22B   | 8            | Top-2                       | Larger MoE variant            |
| DeepSeek-V3     | many experts | sparse activation per token | Large-scale MoE architecture  |
| Qwen3-235B-A22B | 128          | Top-8                       | Open-weight MoE model         |

> The exact expert layout and routing details vary by model, but the core idea is the same: many experts exist, and only a small subset is active for each token.


In [7]:
# Step 1: Implement the Top-K Gating Network


class TopKGating(nn.Module):
    """Router that selects top-k experts for each token."""

    def __init__(self, hidden_dim: int, num_experts: int, k: int = 2):
        super().__init__()
        self.router = nn.Linear(hidden_dim, num_experts, bias=False)
        self.k = k
        self.num_experts = num_experts

    def forward(self, x: torch.Tensor):
        """
        Args:
            x: (batch, seq_len, hidden_dim)
        Returns:
            gate_probs: (batch, seq_len, num_experts) — sparse, only top-k nonzero
            topk_idx:   (batch, seq_len, k) — indices of selected experts
        """
        scores = self.router(x)  # (B, S, E)
        topk_scores, topk_idx = torch.topk(scores, self.k, dim=-1)  # (B, S, k)
        probs = torch.softmax(topk_scores, dim=-1)  # normalize among selected

        # Build sparse gate: only top-k positions are nonzero
        gate = torch.zeros_like(scores)
        gate.scatter_(-1, topk_idx, probs)

        return gate, topk_idx


# Test the router
hidden_dim, num_experts, k = 64, 8, 2
gating = TopKGating(hidden_dim, num_experts, k).to(device)

x = torch.randn(2, 5, hidden_dim, device=device)  # batch=2, seq_len=5
gate, idx = gating(x)

print(f"Input shape: {x.shape}")
print(f"Gate shape:  {gate.shape}  (sparse — only {k} nonzero per token)")
print(f"Top-K idx:   {idx.shape}")
print(f"\nSample gate[0,0]: {gate[0, 0].detach().cpu().tolist()}")
print(f"Selected experts for token [0,0]: {idx[0, 0].tolist()}")
print(f"Gate sums to 1: {gate[0, 0].sum().item():.4f}")

Input shape: torch.Size([2, 5, 64])
Gate shape:  torch.Size([2, 5, 8])  (sparse — only 2 nonzero per token)
Top-K idx:   torch.Size([2, 5, 2])

Sample gate[0,0]: [0.4754689037799835, 0.5245311260223389, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
Selected experts for token [0,0]: [1, 0]
Gate sums to 1: 1.0000


In [8]:
# Step 2: Implement individual Expert (standard FFN with SiLU gating, LLaMA-style)


class Expert(nn.Module):
    """A single FFN expert — identical structure to LLaMA's MLP."""

    def __init__(self, hidden_dim: int, intermediate_dim: int):
        super().__init__()
        self.gate_proj = nn.Linear(hidden_dim, intermediate_dim, bias=False)
        self.up_proj = nn.Linear(hidden_dim, intermediate_dim, bias=False)
        self.down_proj = nn.Linear(intermediate_dim, hidden_dim, bias=False)
        self.act_fn = nn.SiLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.down_proj(self.act_fn(self.gate_proj(x)) * self.up_proj(x))


# Test a single expert
expert = Expert(hidden_dim=64, intermediate_dim=256).to(device)
out = expert(x)
print(f"Expert input: {x.shape} -> output: {out.shape}")
print(f"Expert parameters: {sum(p.numel() for p in expert.parameters()):,}")

Expert input: torch.Size([2, 5, 64]) -> output: torch.Size([2, 5, 64])
Expert parameters: 49,152


In [9]:
# Step 3: Assemble the full MoE Layer


class MoELayer(nn.Module):
    """
    Mixture of Experts layer that replaces the standard FFN.

    Note:
        This is a simple teaching implementation.
        It is mathematically sparse (because only top-k gate weights are nonzero),
        but it is NOT computationally sparse: each expert is still evaluated on all tokens.
        Real MoE systems usually dispatch only the selected tokens to each expert.
    """

    def __init__(self, hidden_dim: int, intermediate_dim: int, num_experts: int = 8, top_k: int = 2):
        super().__init__()
        self.gate = TopKGating(hidden_dim, num_experts, top_k)
        self.experts = nn.ModuleList([Expert(hidden_dim, intermediate_dim) for _ in range(num_experts)])
        self.num_experts = num_experts
        self.top_k = top_k

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (batch, seq_len, hidden_dim)"""
        gate_probs, topk_idx = self.gate(x)  # (B,S,E), (B,S,k)

        output = torch.zeros_like(x)
        for i, expert in enumerate(self.experts):
            expert_weight = gate_probs[:, :, i].unsqueeze(-1)  # (B, S, 1)
            if expert_weight.sum() > 0:
                expert_out = expert(x)  # dense computation for teaching simplicity
                output += expert_weight * expert_out

        return output


# Instantiate and test
moe = MoELayer(hidden_dim=64, intermediate_dim=256, num_experts=8, top_k=2).to(device)
moe_out = moe(x)

total_params = sum(p.numel() for p in moe.parameters())
single_expert_params = sum(p.numel() for p in moe.experts[0].parameters())
active_params = moe.top_k * single_expert_params + sum(p.numel() for p in moe.gate.parameters())

print(f"MoE output shape: {moe_out.shape}")
print(f"Total parameters:  {total_params:>10,}")
print(f"Conceptually active per-token parameters: {active_params:>10,}  ({moe.top_k} experts + router)")
print(f"Fraction of total parameters active per token (conceptually): {active_params / total_params:.2%}")

MoE output shape: torch.Size([2, 5, 64])
Total parameters:     393,728
Conceptually active per-token parameters:     98,816  (2 experts + router)
Fraction of total parameters active per token (conceptually): 25.10%


## 3. Load Balancing in MoE Training

Without any constraint, the router may send too many tokens to the same few "popular" experts, causing:

- Some experts to be over-trained while others are under-used
- Compute / communication imbalance in distributed training

A common idea is to add an auxiliary **load-balancing loss** that encourages more even expert utilization.

One simplified form is:

$$
\mathcal{L}_{\text{balance}} = \alpha \cdot N_E \cdot \sum_{i=1}^{N_E} f_i \cdot P_i
$$

where:

- $f_i$ is the fraction of routing assignments going to expert $i$
- $P_i$ is the average router probability assigned to expert $i$

If routing is more balanced, this quantity is typically closer to its ideal value.

> In the literature, the exact balancing objective varies by model. Some systems use auxiliary losses, while others use alternative routing or balancing strategies.

_Note:_ DeepSeek-V3 is known for using an **auxiliary-loss-free** load-balancing strategy rather than relying on the classic auxiliary balancing loss formulation.


In [10]:
def load_balancing_loss(gate_probs: torch.Tensor, topk_idx: torch.Tensor, num_experts: int) -> torch.Tensor:
    """
    Compute a simple MoE load-balancing loss.

    Args:
        gate_probs: (B, S, E)
            Sparse gate probabilities after top-k selection.
        topk_idx: (B, S, k)
            Selected expert indices for each token.
        num_experts: total number of experts

    Returns:
        Scalar loss whose ideal value is 1.0 under perfectly uniform routing
        for this normalized formulation.
    """
    B, S, E = gate_probs.shape
    K = topk_idx.shape[-1]
    total_assignments = B * S * K  # because each token contributes K expert assignments

    # f_i: fraction of all routing assignments going to expert i
    expert_counts = torch.zeros(E, device=gate_probs.device)
    for i in range(E):
        expert_counts[i] = (topk_idx == i).float().sum()
    f = expert_counts / total_assignments  # now f sums to 1

    # P_i: average gate probability assigned to expert i
    P = gate_probs.mean(dim=(0, 1))  # sums to 1 because each token's sparse gate sums to 1

    loss = E * (f * P).sum()
    return loss


# Demonstrate load balancing
gate, idx = gating(x)
lb_loss = load_balancing_loss(gate, idx, num_experts)

# Check expert distribution
print("=== Expert Load Distribution ===")
total_assignments = idx.numel()
for i in range(num_experts):
    count = (idx == i).sum().item()
    frac = count / total_assignments
    avg_prob = gate[:, :, i].mean().item()
    print(f"  Expert {i}: {count:>3d} assignments ({frac:.2%}), avg prob = {avg_prob:.4f}")

print(f"\nLoad-balancing loss: {lb_loss.item():.4f}")
print(f"Ideal (uniform routing): {1.0:.4f}")

=== Expert Load Distribution ===
  Expert 0:   3 assignments (15.00%), avg prob = 0.1591
  Expert 1:   1 assignments (5.00%), avg prob = 0.0525
  Expert 2:   1 assignments (5.00%), avg prob = 0.0600
  Expert 3:   4 assignments (20.00%), avg prob = 0.1349
  Expert 4:   2 assignments (10.00%), avg prob = 0.0951
  Expert 5:   3 assignments (15.00%), avg prob = 0.1277
  Expert 6:   2 assignments (10.00%), avg prob = 0.1072
  Expert 7:   4 assignments (20.00%), avg prob = 0.2635

Load-balancing loss: 1.1885
Ideal (uniform routing): 1.0000


## 4. Floating-Point Format Comparison

| Format   | Sign | Exponent | Mantissa | Approx Range                          | Common Use                                                 |
| -------- | :--: | :------: | :------: | :------------------------------------ | :--------------------------------------------------------- |
| FP32     |  1   |    8     |    23    | $\pm 3.4 \times 10^{38}$              | Full-precision training, critical evaluation               |
| BF16     |  1   |    8     |    7     | $\pm 3.4 \times 10^{38}$              | Large-scale training (same range as FP32, lower precision) |
| FP16     |  1   |    5     |    10    | $\pm 6.55 \times 10^{4}$              | Mixed-precision training & inference                       |
| FP8 E4M3 |  1   |    4     |    3     | approximately $\pm 448$               | Emerging low-precision inference/training                  |
| FP8 E5M2 |  1   |    5     |    2     | approximately $\pm 5.7 \times 10^{4}$ | Emerging low-precision training                            |

**Trade-off:** More exponent bits → wider dynamic range; more mantissa bits → higher precision.

> The FP8 values above are approximate teaching references. Exact behavior depends on the specific FP8 format, implementation details, and hardware/software support.


In [11]:
# Compare precision across formats
print("=== Floating-Point Format Properties ===")

formats = {
    "FP32": torch.float32,
    "FP16": torch.float16,
    "BF16": torch.bfloat16,
}

x_fp32 = torch.tensor([0.1, 1.0, 100.0, 65504.0, 1e-8], dtype=torch.float32)

print(f"{'Format':>6s} | {'Size (bits)':>11s} | {'eps':>12s} | {'max':>12s} | {'Representation of 0.1':>22s}")
print("-" * 75)
for name, dtype in formats.items():
    info = torch.finfo(dtype)
    val = torch.tensor(0.1, dtype=dtype)
    print(f"{name:>6s} | {info.bits:>11d} | {info.eps:>12.2e} | {info.max:>12.2e} | {val.item():.20f}")

# Demonstrate precision loss
print("\n=== Precision Loss When Casting ===")
large_val = torch.tensor(100000.0, dtype=torch.float32)
for name, dtype in formats.items():
    casted = large_val.to(dtype)
    # Check for overflow
    if casted.isinf():
        print(f"{name}: 100000.0 -> inf (OVERFLOW)")
    else:
        error = abs(casted.float().item() - large_val.item())
        print(f"{name}: 100000.0 -> {casted.float().item()}, error = {error}")

=== Floating-Point Format Properties ===
Format | Size (bits) |          eps |          max |  Representation of 0.1
---------------------------------------------------------------------------
  FP32 |          32 |     1.19e-07 |     3.40e+38 | 0.10000000149011611938
  FP16 |          16 |     9.77e-04 |     6.55e+04 | 0.09997558593750000000
  BF16 |          16 |     7.81e-03 |     3.39e+38 | 0.10009765625000000000

=== Precision Loss When Casting ===
FP32: 100000.0 -> 100000.0, error = 0.0
FP16: 100000.0 -> inf (OVERFLOW)
BF16: 100000.0 -> 99840.0, error = 160.0


## 5. Mixed Precision Training

Mixed precision combines FP32 (master weights, loss scaling) with FP16/BF16 (forward/backward computation) to:

- **Halve** memory usage
- **Double** throughput on Tensor Cores
- Maintain FP32-level training accuracy

Typical flow:

1. Forward pass in FP16/BF16
2. Loss computed in FP32 (with loss scaling for FP16)
3. Backward pass in FP16/BF16
4. Weight update in FP32 (master copy)


In [12]:
# Demonstrate mixed precision with a simple MLP
class SimpleMLP(nn.Module):
    def __init__(self, dim=256):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim * 4)
        self.fc2 = nn.Linear(dim * 4, dim)
        self.act = nn.SiLU()

    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))


dim = 256
model_fp32 = SimpleMLP(dim).to(device)


# Memory comparison
def model_memory(model, dtype=None):
    total = 0
    for p in model.parameters():
        if dtype:
            total += p.numel() * torch.tensor(0, dtype=dtype).element_size()
        else:
            total += p.numel() * p.element_size()
    return total


print("=== Memory Comparison ===")
for name, dtype in [("FP32", torch.float32), ("FP16", torch.float16), ("BF16", torch.bfloat16)]:
    mem = model_memory(model_fp32, dtype)
    print(f"{name}: {mem / 1024:.1f} KB")

# Forward pass in different precisions
x_test = torch.randn(4, dim, device=device)

with torch.no_grad():
    out_fp32 = model_fp32(x_test)
    model_fp16 = SimpleMLP(dim).to(device).half()
    model_fp16.load_state_dict({k: v.half() for k, v in model_fp32.state_dict().items()})
    out_fp16 = model_fp16(x_test.half())

    error = (out_fp32 - out_fp16.float()).abs().mean().item()
    print(f"\nFP32 vs FP16 output mean absolute error: {error:.6f}")

=== Memory Comparison ===
FP32: 2053.0 KB
FP16: 1026.5 KB
BF16: 1026.5 KB

FP32 vs FP16 output mean absolute error: 0.000078


## 6. Quantization Basics

Quantization maps floating-point values to a lower-bit representation, usually to reduce memory usage and improve inference efficiency.

In this notebook, we use a **simple symmetric quantization** example:

$$x_q = \text{round}\left(\frac{x}{\text{scale}}\right)$$

$$\text{scale} = \frac{\max(|x|)}{q_{\max}}$$

and dequantization is:

$$\hat{x} = x_q \cdot \text{scale}$$

This symmetric form is easy to explain and implement.  
In real systems, you may also see **asymmetric quantization**, which additionally uses a `zero_point`.

**Common approaches:**

- **Post-Training Quantization (PTQ)**: Quantize after training, often with a small calibration set
- **Quantization-Aware Training (QAT)**: Simulate quantization effects during training
- **Weight-only INT8/INT4**: Quantize only weights; activations remain in floating point

In practice, quantization introduces approximation error, so there is always a trade-off between:

- **smaller model size / faster inference**
- **numerical accuracy**


In [13]:
def quantize_tensor(x: torch.Tensor, num_bits: int = 8):
    """Symmetric per-tensor quantization.

    Args:
        x: floating-point tensor
        num_bits: target bit-width (e.g. 8 or 4)

    Returns:
        x_q: quantized tensor stored in an integer container
        scale: scale factor for dequantization

    Note:
        For num_bits=4 in this demo, values are still stored in an int8 tensor.
        So this simulates INT4 quantization numerically, but does not bit-pack two 4-bit values into one byte.
    """
    qmin = -(2 ** (num_bits - 1))
    qmax = 2 ** (num_bits - 1) - 1

    x_max = x.abs().max()
    scale = x_max / qmax if x_max > 0 else torch.tensor(1.0, dtype=x.dtype)

    x_q = torch.clamp(torch.round(x / scale), qmin, qmax).to(torch.int8)
    return x_q, scale


def dequantize_tensor(x_q: torch.Tensor, scale: float) -> torch.Tensor:
    """Dequantize: integer tensor -> floating-point approximation."""
    return x_q.float() * scale


# Demo: quantize model weights
weight = model_fp32.fc1.weight.detach().cpu()
print(f"Original weight: shape={weight.shape}, dtype={weight.dtype}")
print(f"  range: [{weight.min():.4f}, {weight.max():.4f}]")
print(f"  memory: {weight.numel() * weight.element_size()} bytes")

# INT8 quantization
w_q8, scale8 = quantize_tensor(weight, num_bits=8)
w_deq8 = dequantize_tensor(w_q8, scale8)
error8 = (weight - w_deq8).abs().mean().item()

print(f"\nINT8 quantized: dtype={w_q8.dtype}")
print(f"  stored memory: {w_q8.numel() * w_q8.element_size()} bytes")
print(f"  compression vs FP32 storage: {weight.element_size() / w_q8.element_size():.0f}×")
print(f"  mean abs error: {error8:.6f}")

# INT4 quantization (simulated numerically, still stored in int8 here)
w_q4, scale4 = quantize_tensor(weight, num_bits=4)
w_deq4 = dequantize_tensor(w_q4, scale4)
error4 = (weight - w_deq4).abs().mean().item()

print("\nINT4 quantized (simulated):")
print(f"  theoretical packed memory: {weight.numel() * 0.5:.0f} bytes")
print(f"  theoretical compression vs FP32 storage: {weight.element_size() / 0.5:.0f}×")
print(f"  mean abs error: {error4:.6f}")

print("\n=== Summary ===")
print("FP32 error: 0.000000")
print(f"INT8 error: {error8:.6f}")
print(f"INT4 error: {error4:.6f}  (higher error, smaller footprint)")

Original weight: shape=torch.Size([1024, 256]), dtype=torch.float32
  range: [-0.0625, 0.0625]
  memory: 1048576 bytes

INT8 quantized: dtype=torch.int8
  stored memory: 262144 bytes
  compression vs FP32 storage: 4×
  mean abs error: 0.000123

INT4 quantized (simulated):
  theoretical packed memory: 131072 bytes
  theoretical compression vs FP32 storage: 8×
  mean abs error: 0.002233

=== Summary ===
FP32 error: 0.000000
INT8 error: 0.000123
INT4 error: 0.002233  (higher error, smaller footprint)


In [14]:
# Impact of quantization on model output
print("=== End-to-End Quantization Impact ===")

# Create a copy of the original model
model_q8 = SimpleMLP(dim).to(device)
model_q8.load_state_dict(model_fp32.state_dict())

with torch.no_grad():
    # Quantize + dequantize fc1 weights
    w1 = model_q8.fc1.weight.data.cpu()
    w1_q, s1 = quantize_tensor(w1, 8)
    model_q8.fc1.weight.data = dequantize_tensor(w1_q, s1).to(device)

    # Quantize + dequantize fc2 weights
    w2 = model_q8.fc2.weight.data.cpu()
    w2_q, s2 = quantize_tensor(w2, 8)
    model_q8.fc2.weight.data = dequantize_tensor(w2_q, s2).to(device)

    # Biases remain in floating point in this simplified demo
    out_orig = model_fp32(x_test)
    out_q8 = model_q8(x_test)

output_error = (out_orig - out_q8).abs().mean().item()
denom = max(out_orig.abs().mean().item(), 1e-12)
relative_error = output_error / denom * 100

print(f"Output mean abs error: {output_error:.6f}")
print(f"Relative error: {relative_error:.2f}%")

print("\nNote:")
print("- This is a simplified demonstration of weight quantization impact.")
print("- We quantize weights, dequantize them back to float, and then run the model.")
print("- Real quantized inference systems may use specialized kernels and more advanced quantization schemes.")

=== End-to-End Quantization Impact ===
Output mean abs error: 0.000817
Relative error: 0.57%

Note:
- This is a simplified demonstration of weight quantization impact.
- We quantize weights, dequantize them back to float, and then run the model.
- Real quantized inference systems may use specialized kernels and more advanced quantization schemes.


## Conclusions

### Technical Concepts Learned

- **MoE Architecture**: Sparse expert selection via Top-K gating, scaling capacity without scaling compute
- **Gating Network**: Router linear layer + Top-K + softmax; `scatter` for sparse gate construction
- **Load Balancing**: Auxiliary loss to prevent expert collapse during training
- **Floating-Point Formats**: FP32, BF16, FP16, and conceptually FP8 differ in numeric range, precision, memory use, and hardware efficiency
- **Mixed Precision**: Lower-precision compute can reduce memory use and improve throughput, while sensitive parts may remain in higher precision
- **Quantization**: Mapping float weights to INT8/INT4 for inference compression; there is a trade-off between model size and numerical accuracy

### Experiment Further

- Modify `top_k` from 2 to 1 and observe the expert distribution change
- Compare per-channel vs per-tensor quantization accuracy
- Apply quantization to a real LLaMA model using `bitsandbytes` or `auto-gptq`
- Implement a capacity factor to limit maximum tokens per expert
- Benchmark MoE vs dense FFN with the same compute budget


---

Copyright (C) 2025 Advanced Micro Devices, Inc. All rights reserved. Portions of this file consist of AI-generated content.
SPDX-License-Identifier: MIT
